# 10. Complexity of Python's Built-in Operations

Knowing Big-O of **algorithms** is not enough. In Python, one innocent line (`x in lst`, `s += ch`, `lst.pop(0)`) can silently turn an O(n) solution into O(n²). Interviewers check this.

**Learning order:** WHY IT MATTERS → LIST → DICT AND SET → DEQUE AND HEAP → STRING → SORT AND SEARCH → HIDDEN COSTS → TIMING PROOF → INTERVIEW THINKING

## 1. List (dynamic array)

| Operation | Time | Note |
|---|---|---|
| `lst[i]`, `lst[i] = x`, `len(lst)` | **O(1)** | |
| `lst.append(x)`, `lst.pop()` | **O(1)** amortized / O(1) | end of the list |
| `lst.pop(0)`, `lst.insert(0, x)`, `lst.insert(i, x)` | **O(n)** | shifts elements |
| `x in lst`, `lst.index(x)`, `lst.remove(x)`, `lst.count(x)` | **O(n)** | linear scan |
| `lst[a:b]` (slice) | **O(b - a)** | creates a **copy** |
| `lst + other`, `lst.copy()`, `list(lst)` | O(n) | |
| `lst.extend(other)` | O(len(other)) | |
| `lst.sort()`, `sorted(lst)` | O(n log n) | Timsort, O(n) extra space |
| `min(lst)`, `max(lst)`, `sum(lst)` | O(n) | |
| `lst.reverse()`, `lst[::-1]` | O(n) | |

## 2. Dict and Set (hash tables)

| Operation | Average | Worst |
|---|---|---|
| `d[k]`, `d[k] = v`, `k in d`, `del d[k]`, `d.get(k)` | **O(1)** | O(n) (many collisions, very rare) |
| `s.add(x)`, `x in s`, `s.remove(x)` | **O(1)** | O(n) |
| iterate all items | O(n) | |
| `s1 \| s2` (union), `s1 & s2` (intersection) | O(n + m), O(min(n, m)) | |
| building `Counter`, `set(lst)`, `dict(...)` | O(n) | |

Keys must be **hashable** (immutable): `int`, `str`, `tuple` of hashables, `frozenset`. A `list` cannot be a key.

## 3. `deque`, `heapq`, `bisect`

| Structure | Operation | Time |
|---|---|---|
| `collections.deque` | `append`, `appendleft`, `pop`, `popleft` | **O(1)** |
| | `dq[i]` in the middle | O(n) |
| `heapq` | `heappush`, `heappop` | O(log n) |
| | `heapify(lst)` | **O(n)** |
| | `heap[0]` (peek) | O(1) |
| | `nsmallest(k, it)`, `nlargest(k, it)` | O(n log k) |
| `bisect` | `bisect_left`, `bisect_right` | O(log n) on a sorted list |
| | `insort` | **O(n)** (the insert shifts elements) |

## 4. String

Strings are **immutable**: every "change" builds a **new** string.

| Operation | Time |
|---|---|
| `s[i]`, `len(s)` | O(1) |
| `s[a:b]` | O(b - a) (copy) |
| `s + t` | O(len(s) + len(t)) |
| `s += ch` in a loop | **O(n²)** overall in the worst case (CPython sometimes optimizes it, never rely on it) |
| `"".join(parts)` | **O(total length)** |
| `ch in s`, `s.find(sub)` | O(n · m) worst case for a substring search |
| `s.split()`, `s.replace(a, b)`, `s.upper()` | O(n) |
| `s == t`, `s < t` | O(min(len(s), len(t))) |
| `s[::-1]` | O(n) |

**Rule:** build strings with a **list + `"".join`**.

## 5. Hidden costs that cost marks in interviews

- `x in some_list` inside a loop → O(n²). Convert to a `set` first.
- `lst.pop(0)` / `lst.insert(0, x)` in a loop → O(n²). Use `deque`.
- Slicing inside recursion (`nums[1:]`) → O(n) copy per call. Pass indexes.
- `lst1 + lst2` in a loop → O(n²). Use `extend`.
- Using `str(n)` repeatedly in a hot loop.
- Passing a big list by value in a recursion is fine (references), but `copy` of it is O(n).
- `sorted(...)` inside a loop. Sort once.
- Counting `len(some_generator)` is impossible; `list(generator)` costs O(n) memory.

In [1]:
import timeit

# 1) membership: list vs set
data_list = list(range(20000))
data_set = set(data_list)
t_list = timeit.timeit(lambda: 19999 in data_list, number=200)
t_set = timeit.timeit(lambda: 19999 in data_set, number=200)
print(t_list > t_set)   # True  (list scans, set hashes)

# 2) front insertion vs append
def build_append(n):
    a = []
    for i in range(n):
        a.append(i)

def build_insert_front(n):
    a = []
    for i in range(n):
        a.insert(0, i)

print(timeit.timeit(lambda: build_insert_front(10000), number=3) > timeit.timeit(lambda: build_append(10000), number=3))   # True

True
True


In [2]:
# 3) list growth is amortized O(1): the capacity jumps only a few times
import sys
sizes, a = [], []
for i in range(1000):
    a.append(i)
    sizes.append(sys.getsizeof(a))
resizes = len(set(sizes))
print(resizes < 40)   # True  (about 30 reallocations for 1000 appends, not 1000)

True


## 6. Space complexity of common things

- A list/set/dict of `n` items: O(n).
- A recursion of depth `d`: O(d) stack.
- `sorted(x)` / slicing / `copy`: O(n) extra. `x.sort()`: O(n) in the worst case for Timsort's merge buffer (practically n/2).
- A 2D table `m × n`: O(m·n).
- Strings you build: O(total length).

## 7. Interview thinking

**Say:** "A list membership test is O(n), so inside a loop this would be O(n²). I'll use a set for O(1) lookups; the extra space is O(n)."

**When asked "what is the complexity of your code?"** count the **loops, recursion, and built-in calls inside them**. Each built-in has its own cost.

**Common mistakes:** treating slicing as O(1), assuming `dict` lookups are O(1) in the worst case without saying "on average", forgetting `sort` is O(n log n) in a "linear" solution, forgetting the O(n) cost of `copy`/`join`.

## Quick Revision

- List: index O(1), end ops O(1), front ops/`in`/`remove` O(n), slice = copy.
- Dict/set: O(1) average. Keys must be hashable.
- `deque` for queues, `heapq` for priority, `bisect` for sorted lists (insert still O(n)).
- Strings: immutable, build with `join`.
- Sort = O(n log n). Hidden O(n) costs hide in slices, `in`, `copy`, `+`.